# 05 — Report and exports

`stackpieces.pipeline.run_all` runs every step from the constraints and writes `results.json`, a self-contained
`report.html` (open it in a browser), STL files and parametric SolidWorks specs. Quick mode takes about 2 minutes; the
full run that produced `results/reference` takes about 15–20 minutes. `compare` lists what moved.

In [1]:
# Setup: works from this folder whether or not the package was pip-installed
import sys, os, json
HERE = os.path.abspath("")
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) == "notebooks" else HERE
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import numpy as np
import matplotlib.pyplot as plt
import stackpieces
print("stackpieces", stackpieces.__version__, "| folder", os.path.relpath(ROOT))
CONSTRAINTS = os.path.join(ROOT, "constraints", "checkers_rot3u.json")
REFERENCE = os.path.join(ROOT, "results", "reference", "results.json")
c = json.load(open(CONSTRAINTS))
QUICK = True   # True: a few minutes, coarse. False: the full reference run (tens of minutes).

stackpieces 0.1.0 | folder ..


In [2]:
from stackpieces.pipeline import run_all, compare
OUT = os.path.relpath(os.path.join(ROOT, "results", "notebook_run"))
res = run_all(CONSTRAINTS, OUT, chosen="S40", quick=QUICK)
print("report:", os.path.join(OUT, "report.html"))

[     0 s] search: enumerating candidates from the constraints


[     1 s] search: 6 candidates, 3 pass the hard checks


[    15 s] finalist S40-b12-s2-t4: printed 50 % at 3.0 mm


[    27 s] finalist S45-b12-s2-t4: printed 50 % at 1.75 mm


[    29 s] curve S40 smooth: 50 % at 5.5 mm (2.4 s)


[    51 s] curve S40 layered: 50 % at 3.1666666666666665 mm (21.9 s)


[    66 s] curve S40 hybrid: 50 % at 4.5 mm (14.2 s)


[    73 s] 3D map S40: 8 s
[    73 s] done: ../results/notebook_run/report.html
report: ../results/notebook_run/report.html


In [3]:
if os.path.exists(REFERENCE):
    ref = json.load(open(REFERENCE))
    for what, a, b, d, flag in compare(res, ref):
        print(("CHECK " if flag else "ok    ") + f"{what:45s} reference {a}  new {b}  diff {d}")
    print("(quick mode uses few releases, so CHECK lines on 50 % points are expected there)")

ok    S40 layered e50 (mm)                          reference 3.4  new 3.1666666666666665  diff -0.23
ok    S40 layered reliable (mm)                     reference 2.5  new 2.0  diff -0.5
ok    S40 smooth e50 (mm)                           reference 5.5  new 5.5  diff 0.0
ok    S40 smooth reliable (mm)                      reference 4.5  new 5.0  diff 0.5
ok    S40 hybrid e50 (mm)                           reference 4.5  new 4.5  diff 0.0
ok    S40 hybrid reliable (mm)                      reference 3.5  new 3.0  diff -0.5
ok    3D S40 place|0.4 cells differing              reference 0  new 0  diff 0
ok    3D S40 slide|0.4 cells differing              reference 0  new 0  diff 0
(quick mode uses few releases, so CHECK lines on 50 % points are expected there)


## Files for printing and for SolidWorks

The STL is a watertight revolve in mm, ready for Bambu Studio (socket on the bed, no supports). The spec is the
partspec JSON format of Tico's SolidWorks build scripts (`revolve` + `revolve_cut`, every size a Global Variable).
For S40 it matches `cad/checker_s40/checker_s40_partspec.json`, which the SolidWorks part was built from.

In [4]:
from stackpieces.geometry import make_piece, PRESETS
from stackpieces.export import write_stl, write_partspec
g = make_piece(PRESETS["S40"])
print(write_stl(os.path.join(OUT, "S40.stl"), g["profile"], name="S40"))
spec = write_partspec(os.path.join(OUT, "S40_partspec.json"), g, "checker_s40")
print(json.dumps(spec["params"], indent=1))

{'path': '../results/notebook_run/S40.stl', 'triangles': 2048, 'volume_mm3': 5657.1926300477, 'watertight': True}
{
 "D": 25.0,
 "HC": 12.0,
 "SH": 2.0,
 "TIP_D": 4.0,
 "CONE_H": 10.13,
 "CLR": 0.5,
 "VCLR": 0.5,
 "CT": 0.5,
 "CB": 0.4,
 "SLOPE": "(D/2 - SH - TIP_D/2) / CONE_H",
 "MOUTH_D": "D - 2*SH + 2*CLR",
 "SOCK_H": "CONE_H + VCLR",
 "ROOF_D": "MOUTH_D - 2*SOCK_H*SLOPE"
}
